<a href="https://colab.research.google.com/github/MarcinPlodzien/deep-learning-for-classical-and-quantum-physics/blob/main/notebooks/lecture-08/exercise-02-stock-market-returns.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

<hr/>
# Deep Learning in Classical and Quantum Physics
2026/2027 <br>
**Author:** Marcin Płodzień <br>
**Exercise:** 02 <br>
**Related labs:** 033, 034, 035, 036
<hr/>

# Exercise 02 — Stock market returns

In Labs 033–036 we analysed and forecast three deterministic signals, the Lorenz-63 convection model, the Mackey–Glass delay equation and the Santa Fe laser. Their autocorrelation functions decay over tens of samples, their spectra have peaks, their delay embeddings have a low dimension, and recurrent networks forecast them with a total capacity $C_{\rm tot}$ above 40 out of 50. In this exercise we apply the same toolbox to a signal of a different kind, the daily closing level of the S&P 500 stock index. The record covers the trading days from 1927 to 2026, 24 671 values. It serves as a control case, the output of a stochastic and adaptive system with no physical attractor.

The price itself is a nonstationary random walk, so we work with the daily log-returns. We first check stationarity, then study the distribution of the returns and their correlations, then test whether they contain deterministic structure, and finally forecast them and their magnitudes with the models of the forecasting labs, from the autoregressive model to recurrent neural networks. Each step is set as a task, Exercise 2.1 to Exercise 2.19, with a stated deliverable, and the surrogate test of Exercise 2.10 is optional. The functions you need are those of the labs, with the same definitions and the same score functions, and the hints name the lab in which each one is introduced.

> **How to run this notebook in Google Colab**
> 1. Go to <https://colab.research.google.com> → *File → Upload notebook*. The first code cell sets `JAX_PLATFORMS=cpu`, and the complete solution runs in about five minutes on a multi-core CPU.
> 2. `jax`, `flax`, `optax`, `statsmodels`, `scipy`, `scikit-learn` and `matplotlib` are pre-installed in Colab.
> 3. The first code cell downloads the data file into a folder `data` next to the notebook.
> 4. Run the cells *in order*. Later cells use variables defined earlier.

In [ ]:
# data files of this notebook, downloaded from the course repository if they are not in ./data
import os, urllib.request
os.makedirs("data", exist_ok=True)
for name in ['sp500_daily_close.txt']:
    if not os.path.exists(f"data/{name}"):
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/MarcinPlodzien/deep-learning-for-classical-and-quantum-physics/main/notebooks/lecture-08/data/{name}", f"data/{name}")

In [ ]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"     # run JAX on the CPU (set before importing jax)

import json, time
from functools import partial
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import scipy.signal
import scipy.stats
from statsmodels.tsa.stattools import acf as sm_acf, adfuller
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
from sklearn.linear_model import Ridge as SkRidge
import warnings

import jax
jax.config.update("jax_enable_x64", True)   # data and linear fits in float64, networks in float32
import jax.numpy as jnp
import optax
import flax
from flax import nnx

# common style of the notebook figures
mpl.rcParams.update({"font.size": 14, "axes.labelsize": 15, "axes.titlesize": 15,
                     "xtick.labelsize": 13, "ytick.labelsize": 13, "legend.fontsize": 12,
                     "figure.titlesize": 16, "lines.linewidth": 2.0,
                     "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False})
# slide figures are drawn at their printed size (5.5 in = full width of a slide), so these are true font sizes
SLIDE_RC = {"font.size": 11, "axes.labelsize": 11, "axes.titlesize": 11, "xtick.labelsize": 11,
            "ytick.labelsize": 11, "legend.fontsize": 11, "lines.linewidth": 1.4, "figure.dpi": 150}

UJBlue, UJNavy, DSOrange, UJGray = "#005CA7", "#002B5C", "#D35A05", "#5A6270"
# one colour and line style per model, as in Lab 036
STYLE = {"persistence": dict(color="0.55", ls=":"), "mean": dict(color="0.2", ls=":"),
         "AR": dict(color="0.35", ls="--"), "MA": dict(color="#8172B2", ls="-."), "NVAR": dict(color=DSOrange, ls="--"),
         "RNN": dict(color=UJBlue, ls="-"), "LSTM": dict(color="#C44E52", ls="-")}

OUT = "figures"
os.makedirs(OUT, exist_ok=True)
def save(fig, name):
    # slide figures are created at their printed size and saved as PDF (slides) and PNG
    for ext in ("pdf", "png"):
        fig.savefig(f"{OUT}/{name}.{ext}", dpi=300, bbox_inches="tight", pad_inches=0.02)

nums = {}                 # every number the text quotes, written to the numbers file at the end
t_start = time.time()
print("jax", jax.__version__, "| flax", flax.__version__, "| optax", optax.__version__)

<hr/>
## 1. Prices and returns

The file contains one closing level $P_k$ per trading day $k$. Prices grow multiplicatively, a move of one percent means the same at any level, so the natural variable is the logarithm. The daily log-return is
$$
r_k=\ln\frac{P_k}{P_{k-1}}=\ln P_k-\ln P_{k-1},
$$
which for small moves is close to the relative change $(P_k-P_{k-1})/P_{k-1}$. The analysis window of this exercise is the last 5000 returns of the record, about the last fifth of it. Section 4 splits the window in time order into a fitting block, its first 3000 returns, a validation block and a test block of 1000 returns each. Every statistic of Sections 1 to 3 is computed on data up to the end of the fitting block, the fitting block of the window or the record up to its end, so that the validation and test blocks do not influence the choice of the forecast variable, of the scaling or of the models. The figures of the whole record are shown for orientation only.

**Exercise 2.1.** Load the closing levels, compute the log-returns of the whole record and keep the last 5000 of them as the analysis window. Print the number of trading days, and for the record up to the end of the fitting block and for the fitting block of the window the mean and standard deviation of the returns, their extreme values and the number of days with zero return. Plot the price on a logarithmic axis, the returns of the whole record and the returns of the window.

In [ ]:
# Exercise 2.1: np.loadtxt, np.log, np.diff; keep r = r_all[-5000:] and the 5001 log-prices behind it.
# Plot P_k with semilogy and mark the analysis window with axvspan.
...

### Random walk and differencing

A random walk with drift is the simplest model of the log-price,
$$
\ln P_k=\ln P_{k-1}+\mu+\sigma\eta_k,\qquad \eta_k\ \text{independent}\ \mathcal N(0,1),
$$
with drift $\mu$ and daily volatility $\sigma$. It is the autoregressive model AR(1) $\ln P_k=c+\phi\ln P_{k-1}+\varepsilon_k$ at $\phi=1$, a unit root of the characteristic polynomial $\Phi_1(z)=1-\phi z$ (Lab 034), and in the notation of the autoregressive integrated moving-average models it is ARIMA(0,1,0) with drift. One difference, $(1-B)\ln P_k=r_k=\mu+\sigma\eta_k$, turns it into white noise with mean $\mu$. A model of this kind has no stationary mean, and its variance grows linearly in time. Forecasting the level then rewards persistence, $\widehat{\ln P}_{k+1}=\ln P_k$, without saying anything about the dynamics.

**Exercise 2.2.** Fit AR(1) with a constant to the 3001 log-prices of the fitting block by least squares in JAX and with `statsmodels` `AutoReg`, and report $\hat\phi$ with its standard error. Simulate three random walks with the drift and volatility of the fitting block, starting at the same log-price, and plot them together with the data. Compute the squared correlation $C(1)$ of persistence for the log-price and for the returns over the last 1000 days of the fitting block.

In [ ]:
# Exercise 2.2: the AR(1) least-squares fit as in the AR section of Lab 034 (design matrix rows (1, ln P_{k-1})).
# Random walk: lp[0] + cumsum(mu + sigma * eta) with eta from jax.random.normal.
...

### Stationarity

A process is stationary when its mean and autocovariance do not depend on the origin of time (Lab 033). A quick check slides a window of $W=250$ trading days, about one trading year, over the record and computes the rolling mean $\bar r^{(250)}_k$ and the rolling standard deviation $\sigma^{(250)}_k$ of the returns $r_{k-249},\dots,r_k$.

A formal test of the unit root is the augmented Dickey–Fuller (ADF) test [D. A. Dickey and W. A. Fuller, J. Am. Stat. Assoc. **74**, 427 (1979), [doi:10.1080/01621459.1979.10482531](https://doi.org/10.1080/01621459.1979.10482531); S. E. Said and D. A. Dickey, Biometrika **71**, 599 (1984), [doi:10.1093/biomet/71.3.599](https://doi.org/10.1093/biomet/71.3.599)]. It fits by least squares the regression
$$
\begin{aligned}
\Delta y_k&=\alpha+\beta\,y_{k-1}+\sum_{i=1}^{p}\gamma_i\,\Delta y_{k-i}+\varepsilon_k,\\
\Delta y_k&=y_k-y_{k-1},
\end{aligned}
$$
where the lagged differences absorb short-range correlations of $\Delta y$. A unit root means $\beta=0$, and a stationary AR process has $\beta<0$. The null hypothesis is the unit root, $H_0:\beta=0$, against $\beta<0$. The test statistic is the $t$-ratio $\hat\beta/{\rm se}(\hat\beta)$. Under $H_0$ it does not follow the Student distribution, and its critical values come from the Dickey–Fuller distribution, which `statsmodels.tsa.stattools.adfuller` tabulates and returns with a $p$-value. A strongly negative statistic rejects the unit root. Option `regression="ct"` adds a linear trend $\delta k$ to the regression, and `autolag="AIC"` chooses $p$ by the Akaike information criterion.

**Exercise 2.3.** Plot the rolling mean and the rolling standard deviation of the returns over the record up to the end of the fitting block, and report the ratio of the largest to the smallest rolling standard deviation. Implement the ADF statistic in JAX for a fixed number of lags $p$ and check it against `adfuller(..., maxlag=p, autolag=None)`. Then run `adfuller` with `autolag="AIC"` on the log-price (with constant, and with constant and trend), on the returns and on $|r_k|$, for the record up to the end of the fitting block and for the fitting block, and tabulate the statistic, the $p$-value, the number of lags and the 5% critical value.

In [ ]:
# Exercise 2.3a: np.lib.stride_tricks.sliding_window_view(x, 250).mean(axis=1) and .std(axis=1).
...

In [ ]:
# Exercise 2.3b: build X = [1, y_{k-1}, dy_{k-1}, ..., dy_{k-p}], solve with jnp.linalg.lstsq, se from s^2 (X^T X)^{-1}.
# adfuller returns (stat, p-value, used lags, nobs, critical values, icbest).
...

<hr/>
## 2. Distribution, correlations and spectra

### Heavy tails

We standardise the returns of the fitting block, $x_k=(r_k-\bar r)/\sigma_r$, with their mean $\bar r$ and standard deviation $\sigma_r$. A Gaussian variable has skewness $\langle x^3\rangle=0$ and excess kurtosis $\kappa=\langle x^4\rangle-3=0$. A positive $\kappa$ means more probability in the tails and in the centre than a Gaussian of the same variance. The Student $t$ distribution with $\nu$ degrees of freedom has power-law tails, $p(x)\propto|x|^{-(\nu+1)}$ for large $|x|$, and it is a common one-parameter model of heavy tails [R. Cont, Quant. Finance **1**, 223 (2001), [doi:10.1080/713665670](https://doi.org/10.1080/713665670)].

**Exercise 2.4.** Compute the skewness and the excess kurtosis of the fitting block in JAX and compare with `scipy.stats`. Fit a Student $t$ distribution with `scipy.stats.t.fit`. For $a=3,4,5,6$ compare the fraction of days with $|x_k|>a$ with the Gaussian value $2\,\Phi(-a)$, $\Phi$ the standard normal distribution function, and with the fitted $t$ distribution. Plot the histogram on a logarithmic axis and the tail probability $P(|x_k|>a)$ against $a$.

In [ ]:
# Exercise 2.4: moments with jnp.mean of powers of the centred sample; scipy.stats.skew, scipy.stats.kurtosis (excess);
# scipy.stats.t.fit(x) returns (nu, loc, scale); Gaussian tail 2 * scipy.stats.norm.sf(a).
...

### Autocorrelation of returns and of their size

The autocorrelation function (ACF) is estimated as in Lab 033,
$$
\hat\rho(k)=\frac{\sum_{n=0}^{N-1-k}(x_n-\bar x)(x_{n+k}-\bar x)}{\sum_{n=0}^{N-1}(x_n-\bar x)^2},
$$
with $\bar x$ the mean of the $N$ values. The sum of $N-k$ products is divided by the sum of all $N$ squares, which is the convention of `statsmodels` with `adjusted=False`. For $N$ independent samples the estimates at $k\ge1$ scatter around zero with standard deviation close to $1/\sqrt N$, so about 95% of them lie in the band $\pm1.96/\sqrt N$. The ACF of $r_k$ measures linear predictability of the returns. The ACF of $|r_k|$ and of $r_k^2$ measures whether large moves follow large moves, whatever their sign, which is called volatility clustering.

**Exercise 2.5.** Compute $\hat\rho(k)$ for $k=0,\dots,200$ of $r_k$, $|r_k|$ and $r_k^2$ of the fitting block with the JAX function of Lab 033 and check it against `statsmodels`. Report the values at a few lags and the number of lags in $1,\dots,200$ outside the 95% band. Plot the ACF of $r_k$ at small lags and the three ACFs up to lag 200.

In [ ]:
# Exercise 2.5: acf_jax from Lab 033 (vmap over the lags), statsmodels acf(x, nlags, adjusted=False, fft=False).
...

### Power spectra

By the Wiener–Khinchin theorem (Lab 033) an ACF that vanishes at all $k\ge1$ corresponds to a flat power spectral density (PSD). For a white noise of unit variance and unit sampling interval the one-sided PSD is $S(f)=2$ at every frequency. A slowly decaying ACF puts excess power at low frequencies.

**Exercise 2.6.** Compute the Welch estimate (Hann window, segments of $L=512$ days, 50% overlap) of the PSD of the standardised $r_k$ and of the standardised $|r_k|$ of the fitting block with the JAX function of Lab 033, check it against `scipy.signal.welch`, and plot both on logarithmic axes. Quantify the low-frequency excess as the ratio of the mean $\hat S$ over the five lowest nonzero frequencies to the median $\hat S$ above $f=0.1$ cycles per day.

In [ ]:
# Exercise 2.6: welch_jax from Lab 033 (frequencies in cycles per trading day, dt = 1).
...

<hr/>
## 3. Tests for deterministic structure

The chaotic signals of Lab 033 have a delay embedding of low dimension, they reject a linear Gaussian null model in a surrogate test, and their nearby trajectories separate exponentially at a rate $\lambda_1$, the largest Lyapunov exponent. We now run the same diagnostics on the returns. Two reference signals help to read the results, the Lorenz $x$ signal of Lab 033 (4000 samples, $\Delta t=0.02$), whose answers we know, and the returns in random order, which keep the distribution and destroy every time dependence.

**Exercise 2.7.** Copy from Lab 033 the Lorenz-63 generator, the delay embedding, the nearest-neighbour search, the mutual information $I(\ell)$ and the false-nearest-neighbour (FNN) fractions of the ratio test. Generate the Lorenz $x$ reference and a random permutation of the standardised returns of the fitting block. All diagnostics of this section use the 3000 returns of the fitting block.

In [ ]:
# Exercise 2.7: copy lorenz_rhs, rk4_step, lorenz_trajectory, delay_embed, nearest_neighbour, mutual_information,
# first_local_min and fnn_fractions (ratio test only) from Lab 033; shuffle with np.random.default_rng(0).permutation.
...

### Delay and embedding dimension

Lab 033 chose the delay $\ell$ at the first minimum of $I(\ell)$ and the dimension $m$ at the point where the FNN fraction drops below 5%. The histogram estimate of $I(\ell)$ uses 16 bins of equal width over the range of the data. With heavy tails almost all returns fall into the two or three central bins. The mutual information is invariant under a monotone change of variables, so we also compute it from the ranks $u_k={\rm rank}(x_k)/N$, which have a uniform distribution.

**Exercise 2.8.** Compute $I(\ell)$ for $\ell=0,\dots,50$ of the returns with equal bins and with ranks, of the shuffled returns (ranks) and of Lorenz $x$, and report $I(1)$, $I(10)$, the mean over $\ell=20,\dots,50$ and the first local minimum. Compute the FNN fractions for $m=1,\dots,10$ of the returns and of the shuffled returns with $\ell=1$ and of Lorenz $x$ with $\ell=10$, and plot both diagnostics.

In [ ]:
# Exercise 2.8: ranks with scipy.stats.rankdata(x) / N; fnn_fractions(x, lag) as in Lab 033 with Theiler window 50.
...

### Lyapunov exponent and Kaplan–Yorke dimension

The Rosenstein estimate of $\lambda_1$ follows the mean logarithmic distance $\langle\ln d(k)\rangle$ of initially nearest neighbours $k$ steps later and reads $\lambda_1$ from the slope of a straight part of this curve (Lab 033). The Sano–Sawada estimate fits local Jacobians to the neighbourhoods of the delay vectors and returns a whole spectrum $\lambda_1\ge\dots\ge\lambda_m$, from which the Kaplan–Yorke dimension $D_{\rm KY}=j+\Sigma_j/|\lambda_{j+1}|$ follows, $\Sigma_j=\lambda_1+\dots+\lambda_j$ and $j$ the largest index with $\Sigma_j\ge0$. Both algorithms return numbers for any input. Whether the numbers mean anything depends on the assumptions behind them, a deterministic low-dimensional flow sampled densely enough that nearby delay vectors evolve similarly.

**Exercise 2.9.** Compute the Rosenstein curve of the returns and of the shuffled returns ($m=4$, $\ell=1$, Theiler window 50, $k\le50$) and of Lorenz $x$ ($m=4$, $\ell=10$, $k\le150$). Fit slopes over $k=0$–1, 0–5, 5–20 and 20–50 for the returns and compare with the slope of Lorenz $x$ over $k=25$–100. Compute the Sano–Sawada spectrum ($m=4$, $K=63$ neighbours) and $D_{\rm KY}$ for the returns, the shuffled returns and Lorenz $x$. Plot the Rosenstein curves.

In [ ]:
# Exercise 2.9: rosenstein_curve, slope, local_jacobian_spectrum and kaplan_yorke from Lab 033.
...

### Surrogate test (optional)

This exercise is optional, and no later exercise uses its results. The null hypothesis of the surrogate test of Lab 033 (Appendix A) is a stationary linear Gaussian process observed through a static monotone function. Iterative amplitude-adjusted Fourier-transform (IAAFT) surrogates realise it with the exact value distribution and nearly the same spectrum [T. Schreiber and A. Schmitz, Phys. Rev. Lett. **77**, 635 (1996), [doi:10.1103/PhysRevLett.77.635](https://doi.org/10.1103/PhysRevLett.77.635)]. The first statistic is the normalised error $E={\rm rms}(s_{n+1}-s_{j+1})/\sigma_s$ of the nearest-neighbour forecast in the delay space, with $\mathbf y_j$ the nearest neighbour of $\mathbf y_n$ outside the Theiler window, and a one-sided test that rejects $H_0$ when the data error is the smallest of 20 values (19 surrogates). A second statistic targets volatility clustering,
$$
A=\sum_{k=1}^{20}\hat\rho_{|x|}(k),
$$
the sum of the first 20 ACF values of $|x_k|$, with $H_0$ rejected when the data value is the largest of the 20.

**Optional exercise 2.10.** Generate 19 IAAFT surrogates of the standardised returns and of Lorenz $x$. Run the test with $E$ on the returns ($m=4$, $\ell=1$) and on Lorenz $x$ ($m=4$, $\ell=10$), and the test with $A$ on the returns. Report the data value, the range of the surrogate values, the rank and $z=(E_{\rm data}-\bar E_{\rm surr})/\sigma_{\rm surr}$. Repeat the returns test with $E$ for $m=2,6,8$.

In [ ]:
# Optional exercise 2.10: iaaft and nn_prediction_error from Lab 033 (vmap the surrogate generator over 19 keys);
# A = acf_jax(|x - mean|, 20)[1:].sum(); rank = 1 + number of surrogate values more extreme than the data.
...

<hr/>
## 4. Forecasting the returns

We adapt the protocol of Labs 034 and 036 to the window of 5000 returns. The split is chronological, 60/20/20. The first 3000 returns form the fitting (training) block, the next 1000 the validation block and the last 1000 the test block. The returns are standardised, $s_k=(r_k-\mu_{\rm tr})/\sigma_{\rm tr}$, with the mean and standard deviation of the fitting block. The labs rescale to $[0,1]$ with the minimum and maximum instead, which for heavy-tailed data would squeeze almost all values into a narrow band set by a few extreme days. Every model reads at most the window $\mathbf v_k=(s_{k-63},\dots,s_k)$ of $L=64$ values and returns the leads $\tau=1,\dots,50$ directly. The fitting origins are $k=63,\dots,2949$, the validation origins $k=2999,\dots,3949$ and the test origins $k=3999,\dots,4949$. The forecast from the origin $k$ uses only values up to $s_k$, the targets of the fitting origins end at return 2999, before the validation block, and those of the validation origins end at return 3999, before the test block, so no input window or target used for fitting or for a choice reaches the test block.

Persistence and the mean of the fitting block are the baselines. AR and NVAR are ridge regressions on the window, with fixed features and a fitted linear readout. MA forecasts from its last eight one-step errors, which its filter computes from all values up to the origin. The RNN and the LSTM network start from a zero hidden state at the oldest value of the window, and all their weights, recurrence and readout, are learned by gradient descent. The mean baseline is a reference level computed over a fixed period, the fitting block. Its NMSE on the test block is $1+(\bar s_{\rm fit}-\bar s_{\rm test})^2/\sigma^2_{\rm test}$ and exceeds 1 whenever the mean of the test block differs from that of the fitting block.

The scores are those of Lab 034, $\mathrm{NMSE}(\tau)$, the capacity $C(\tau)=r(\tau)^2$ and $C_{\rm tot}=\sum_{\tau=1}^{50}C(\tau)$. For returns the sign of the move matters to a trader, so we add the directional accuracy
$$
\mathrm{DA}(\tau)=\frac1n\sum_{k}\mathbb 1\big[(\hat r_{k+\tau}>0)=(r_{k+\tau}>0)\big],
$$
the fraction of the $n$ test origins at which the forecast and the true return agree on up or down. If the $n$ hits were independent with probability $q$, their number would be binomial and DA would have the standard error $\sqrt{q(1-q)/n}$, which for $n=951$ is about 0.016. A forecast that always says up reaches the fraction of up days, which is above 0.5 for a rising index, so this fraction is the baseline for DA.

**Exercise 2.11.** Set up the standardised returns, the origins and the windows. Copy the score function of Lab 034 and add the directional accuracy with its standard error, computed on the raw returns $r=\mu_{\rm tr}+\sigma_{\rm tr}s$. Score persistence, $\hat s_{k+\tau}=s_k$, and the mean of the fitting block. Report the fraction of up days among the test targets.

In [ ]:
# Exercise 2.11: windows() as in Lab 036; scores and scores_all (vmap over the leads) from Lab 034;
# DA from the signs of mu_tr + sd_tr * prediction and mu_tr + sd_tr * target.
...

### AR and NVAR

The direct autoregressive (AR) model and the nonlinear vector autoregression (NVAR) are ridge regressions on the delay vector $(s_k,\dots,s_{k-p+1})$ and on the delay vector together with its quadratic monomials (Lab 034, Lab 036). In the labs the order $p$ and the ridge strength $\beta$ were chosen by the largest validation $C_{\rm tot}$. On a signal without structure this choice rewards chance correlations, because $C$ ignores the sign and the scale of the forecast. Here we choose $(p,\beta)$ by the smallest validation NMSE averaged over the 50 leads, from $p\in\{1,2,4,8,16,32,64\}$ for AR, $p\in\{2,4,8,12,16\}$ for NVAR and $\beta\in\{10^{-4},10^{-2},1,10^{2},10^{4}\}$.

**Exercise 2.12.** Check the one-step AR(5) readout with a negligible penalty against `statsmodels` `AutoReg` on the training block, and the ridge solution against `sklearn.linear_model.Ridge`. Fit the direct AR and NVAR models with the selection above and report $p$, $\beta$, $C_{\rm tot}$, $C(1)$, $\mathrm{NMSE}(1)$ and $\mathrm{DA}(1)$ on the test block.

In [ ]:
# Exercise 2.12: ridge_fit, delay_features, nvar_features from Lab 036; selection loop by the validation NMSE.
...

### Moving-average model

The moving-average model MA($q$) writes the signal as its mean plus a weighted sum of the present and the last $q$ innovations, $s_k=\mu+\varepsilon_k+\sum_{j=1}^{q}\theta_j\varepsilon_{k-j}$ (Lab 034). Its autocorrelation vanishes beyond lag $q$, so it describes a short memory of the shocks. We fit MA(8) by maximum likelihood with `statsmodels`, `ARIMA(order=(0, 0, 8), trend="c")`, on the first 4000 standardised returns, as Lab 036 fits it on its training and validation blocks. No setting of this model is chosen on the validation block, and the test block is not used. The one-step prediction errors $e_k=s_k-\hat s_k$ of the fitted model, computed over the whole record with `apply`, give the forecast
$$
\hat s_{k+\tau}=\mu+\sum_{j=\tau}^{q}\theta_j\,e_{k+\tau-j},
$$
which uses only errors up to the origin $k$ and equals the mean $\mu$ for $\tau>q$.

**Exercise 2.13.** Fit MA(8) to the standardised returns, report $\mu$, the coefficients $\theta_j$ with their standard errors, and compare the forecast from the first test origin with the `forecast` method of `statsmodels`. Report $C_{\rm tot}$, $C(1)$, $\mathrm{NMSE}(1)$ and $\mathrm{DA}(1)$ on the test block.

In [ ]:
# Exercise 2.13: ARIMA(s[:N_TRAIN], order=(0, 0, 8), trend='c').fit(); e = fit.apply(s).resid;
# forecast as in the static-model comparison of Lab 036; fit.apply(s[:k + 1]).forecast(50) for the check.
...

### Recurrent networks

The recurrent neural network (RNN) updates a hidden state $\mathbf h_k\in\mathbb R^{n_h}$ by $\mathbf h_k=\tanh(W_h\mathbf h_{k-1}+W_xs_k+\mathbf b)$, and the long short-term memory (LSTM) network adds a cell state controlled by forget, input and output gates (Lab 036). We use the cells of Lab 036 with $n_h=32$ hidden units, run over the window from $\mathbf h=0$, a linear readout of the last hidden state to the 50 leads, and the training loop with Adam (learning rate $5\times10^{-3}$, minibatches of 128 windows), gradient clipping at norm 1 and early stopping on the validation loss with patience 20 and at most 250 epochs. The hidden state is reset to zero for every window, so the networks are functions of the 64 values of the window, and all weights, recurrence and readout, are learned. A claim about a network must hold for more than one random initialisation, so both networks are trained with the seeds 0, 1 and 2.

**Exercise 2.14.** Train the RNN and the LSTM network on the returns for the three seeds and report the number of epochs, the test $C_{\rm tot}$, $\mathrm{NMSE}(1)$ and $\mathrm{DA}(1)$ for each seed. Use seed 0 in the comparison table.

In [ ]:
# Exercise 2.14: RNNCell, LSTMCell, Recurrent, fit_network and net_data from Lab 036; loop over seeds 0, 1, 2.
...

### Significance

A model that is independent of the targets still produces a nonzero $C(\tau)$ on a finite test block. For $n$ independent pairs the expected squared correlation is close to $1/(n-1)$, so $C_{\rm tot}\approx50/(n-1)\approx0.05$ without any skill. A permutation test makes this concrete for each model. We randomly reorder the forecast origins, which destroys the pairing of forecasts and targets and keeps both distributions, recompute $C_{\rm tot}$ 500 times and read off the 95th percentile. A model whose test $C_{\rm tot}$ lies below this value shows no evidence of skill at the 5% level. The permutation also destroys the serial correlation of the forecasts, which for the returns is weak.

**Exercise 2.15.** Collect for all models the test $C_{\rm tot}$ with the 95th percentile of its permutation null, $C(1)$, $\mathrm{NMSE}(1)$ and $\mathrm{DA}(1)$ with its standard error and its difference from the fraction of up days. Plot $\mathrm{DA}(\tau)$ with the band of two standard errors around the always-up baseline.

In [ ]:
# Exercise 2.15: vmap a function that permutes the rows of the prediction array (jax.random.permutation)
# and returns the sum of C over the leads; np.quantile(null, 0.95).
...

<hr/>
## 5. Forecasting the volatility

The ACF of $|r_k|$ in Exercise 2.5 shows that the size of tomorrow's move depends on the sizes of the past moves. Econometrics models this conditional variance with the autoregressive conditional heteroskedasticity (ARCH) model [R. F. Engle, Econometrica **50**, 987 (1982), [doi:10.2307/1912773](https://doi.org/10.2307/1912773)] and its generalisation GARCH [T. Bollerslev, J. Econometrics **31**, 307 (1986), [doi:10.1016/0304-4076(86)90063-1](https://doi.org/10.1016/0304-4076(86)90063-1)], in which $r_k=\sigma_k\eta_k$ and $\sigma_k^2$ is a linear function of past $r^2$ and past $\sigma^2$. We stay with the models of the labs and forecast the volatility proxy $|r_k|$ itself, standardised with the mean and standard deviation of its fitting block, the first 3000 values, with the same windows, origins, selection rules and seeds as for the returns.

**Exercise 2.16.** Forecast $|r_{k+\tau}|$ with persistence, the training mean, AR, MA, NVAR, the RNN and the LSTM network, and report $C_{\rm tot}$, $C(\tau)$ and $\mathrm{NMSE}(\tau)$ at $\tau=1,5,20$, with the network $C_{\rm tot}$ for the three seeds. Check how stable the result is by computing $\mathrm{NMSE}(1)$ in five consecutive sub-blocks of the test origins, each against its own mean. Plot $\mathrm{NMSE}(\tau)$.

In [ ]:
# Exercise 2.16: reuse ridge_model, ma_forecaster and net_forecaster with returns=False (no directional accuracy).
...

<hr/>
## 6. One-step forecasts and capacity curves

**Exercise 2.17.** Plot for every model except the training mean the forecast one step ahead, $\tau=1$, in the style of Labs 034 and 036, with one column for the returns and one for the volatility. The upper panels show the standardised target and the forecast on the first 100 test days, and the lower panels show the forecast against the target on all test days, with $C(1)$ and the one-step NMSE. Describe what the figures show.

In [ ]:
# Exercise 2.17: one figure per model (persistence, AR, MA, NVAR, RNN, LSTM), columns returns and |r_k|,
# upper row target and forecast on the first 100 test days, lower row forecast against target with C and NMSE.
...

Lab 036 scored the same models on chaotic signals, the Lorenz $x$ signal, the Mackey–Glass signals and the Santa Fe laser, with $N=4000$ samples, a test block of 1000 samples and leads up to 50. The protocols differ in the selection criterion and the scaling, so the comparison concerns orders of magnitude.

**Exercise 2.18.** Plot the capacity curves $C(\tau)$, $\tau=1,\dots,50$, of the six models for the returns and for the volatility. Print the table of the test $C_{\rm tot}$ for both, next to the values of Lab 036 for Lorenz $x$, Mackey–Glass at $\tau_{\rm MG}=25$ and the Santa Fe laser, and discuss it in the light of Sections 2 and 3.

In [ ]:
# Exercise 2.18: C(tau) of all models for r_k (logarithmic axis) and |r_k|; test C_tot of Lab 036
# (persistence, AR, MA, NVAR, RNN, LSTM; no MA value for Lorenz x) for Lorenz x, MG 25 and Santa Fe next to your values.
...

### References

- D. A. Dickey and W. A. Fuller, Distribution of the estimators for autoregressive time series with a unit root, J. Am. Stat. Assoc. **74**, 427 (1979), [doi:10.1080/01621459.1979.10482531](https://doi.org/10.1080/01621459.1979.10482531).
- S. E. Said and D. A. Dickey, Testing for unit roots in autoregressive-moving average models of unknown order, Biometrika **71**, 599 (1984), [doi:10.1093/biomet/71.3.599](https://doi.org/10.1093/biomet/71.3.599).
- R. Cont, Empirical properties of asset returns. Stylized facts and statistical issues, Quant. Finance **1**, 223 (2001), [doi:10.1080/713665670](https://doi.org/10.1080/713665670).
- R. F. Engle, Autoregressive conditional heteroscedasticity with estimates of the variance of United Kingdom inflation, Econometrica **50**, 987 (1982), [doi:10.2307/1912773](https://doi.org/10.2307/1912773).
- T. Bollerslev, Generalized autoregressive conditional heteroskedasticity, J. Econometrics **31**, 307 (1986), [doi:10.1016/0304-4076(86)90063-1](https://doi.org/10.1016/0304-4076(86)90063-1).
- T. Schreiber and A. Schmitz, Improved surrogate data for nonlinearity tests, Phys. Rev. Lett. **77**, 635 (1996), [doi:10.1103/PhysRevLett.77.635](https://doi.org/10.1103/PhysRevLett.77.635).
- M. T. Rosenstein, J. J. Collins and C. J. De Luca, A practical method for calculating largest Lyapunov exponents from small data sets, Physica D **65**, 117 (1993), [doi:10.1016/0167-2789(93)90009-P](https://doi.org/10.1016/0167-2789(93)90009-P).
- M. Sano and Y. Sawada, Measurement of the Lyapunov spectrum from a chaotic time series, Phys. Rev. Lett. **55**, 1082 (1985), [doi:10.1103/PhysRevLett.55.1082](https://doi.org/10.1103/PhysRevLett.55.1082).

<hr/>
## Summary

**Exercise 2.19.** Write a short summary in connected prose. State which properties of the returns you established, with numbers, which diagnostics of deterministic chaos fail on this signal and why, which forecasts beat their baselines and by how much, and what a claim of predictability of a financial time series requires.